# Step 05.4 — Gradient Descent Loop

5.3 已经能算出 gradient。现在第一次真正修改 parameter。

核心公式：

```text
parameter_new = parameter_old - learning_rate * gradient
```

这节只解决四件事：

1. 为什么是减去 gradient；
2. learning rate 控制什么；
3. 怎样把 forward / loss / gradient / update 组成循环；
4. 为什么 learning rate 太大会发散。

## 1. 为什么沿着 `-gradient` 更新？

继续使用上一节的极小模型：

```text
prediction = w*x
x = 2
target = 10
loss = (w*x-target)^2
```

当：

```text
w = 3
```

我们已经算出：

```text
loss = 16
dL/dw = -16
```

gradient 为负，说明 **w 往更大的方向走，loss 会下降**。

因此统一写成：

```text
w_new = w_old - lr * gradient
```

如果 gradient 是负数，减去负数就会增加 w；如果 gradient 是正数，就会减小 w。

所以 `-gradient` 自动给出局部下降方向。

In [ ]:
import mlx.core as mx

x = mx.array(2.0)
target = mx.array(10.0)

def loss_fn(w):
    prediction = w * x
    return mx.square(prediction - target)

loss_and_grad_fn = mx.value_and_grad(loss_fn)

w = mx.array(3.0)
loss, gradient = loss_and_grad_fn(w)
mx.eval(loss, gradient)

print("w       :", w)
print("loss    :", loss)
print("gradient:", gradient)

现在选择：

```text
learning_rate = 0.1
```

那么：

```text
w_new = 3 - 0.1*(-16)
      = 4.6
```

更新以后：

```text
prediction = 4.6*2 = 9.2
loss = (9.2-10)^2 = 0.64
```

一次更新就把 loss 从 16 降到了 0.64。

In [ ]:
learning_rate = 0.1

w_new = w - learning_rate * gradient
new_loss = loss_fn(w_new)

mx.eval(w_new, new_loss)

print("old w   :", w)
print("new w   :", w_new)
print("old loss:", loss)
print("new loss:", new_loss)

## 2. 把这件事重复：第一个真正的 Training Loop

训练最基本的闭环其实就是：

```text
当前 parameter
      ↓
   forward
      ↓
     loss
      ↓
   gradient
      ↓
parameter update
      ↓
下一轮
```

现在把它写成循环。

In [ ]:
w = mx.array(3.0)
learning_rate = 0.1

for step in range(8):
    loss, gradient = loss_and_grad_fn(w)

    # Gradient Descent update
    w = w - learning_rate * gradient

    mx.eval(w, loss)

    print(
        f"step={step:2d}  "
        f"loss={loss.item():.6f}  "
        f"w={w.item():.6f}"
    )

你应该看到：

```text
loss: 16 -> 0.64 -> 0.0256 -> ... -> 接近 0
w   : 3  -> 4.6  -> 4.92   -> ... -> 接近 5
```

为什么最终 `w -> 5`？

因为正确答案要求：

```text
w*x = target
w*2 = 10
w = 5
```

也就是说训练过程自己通过 gradient 一步一步把 parameter 从 3 推到了接近 5。

## 3. Learning Rate 到底控制什么？

公式：

```text
w_new = w - lr * gradient
```

`gradient` 给方向和局部坡度；`learning_rate` 决定我们沿这个方向迈多大一步。

可以粗略理解：

```text
lr 很小 -> 每步很谨慎，通常慢
lr 合适 -> 较快接近低 loss 区域
lr 太大 -> 一步跨过头，甚至来回震荡或越来越远
```

In [ ]:
def run_descent(lr, steps=6):
    w = mx.array(3.0)
    losses = []

    for _ in range(steps):
        loss, gradient = loss_and_grad_fn(w)
        w = w - lr * gradient
        mx.eval(w, loss)
        losses.append(loss.item())

    return w.item(), losses


for lr in [0.01, 0.1, 0.3]:
    final_w, losses = run_descent(lr)
    print(f"lr={lr}")
    print("  losses :", [round(v, 6) for v in losses])
    print("  final w:", round(final_w, 6))

对这个特定问题：

- `lr=0.01`：通常下降，但比较慢；
- `lr=0.1`：快速收敛；
- `lr=0.3`：可能越走越远，loss 增长。

这不是说 `0.1` 在所有神经网络里都最好。

Learning rate 的合适尺度取决于 loss landscape、模型、优化器、batch、归一化等很多因素。

这个实验只是让你看到：**gradient 给出的只是局部方向；步子迈多大仍然需要人为/优化器控制。**

## 4. Gradient Descent、SGD、Adam 暂时怎么区分？

我们当前这个例子每一步都使用同一个完整训练样本，所以更准确地叫 gradient descent。

以后如果数据很多：

- 用全部数据算一次 gradient：full-batch gradient descent；
- 随机用一个样本：stochastic gradient descent（SGD）；
- 用一个小 batch：mini-batch SGD，这是深度学习中更常见的训练方式。

而 Adam 不是“没有 gradient”，它仍然使用 gradient，只是根据历史的一阶/二阶统计量决定每个 parameter 实际迈多大步。

MiniGPT 后面会使用 Adam 类优化器，但现在先把最朴素的 gradient descent 闭环吃透。

### 5.4 最终结论

- gradient 指向 loss 局部上升最快的方向；
- 所以减小 loss 时沿 `-gradient` 更新；
- learning rate 决定更新步长；
- 训练循环最小闭环是 `forward -> loss -> gradient -> update -> repeat`；
- learning rate 太小会慢，太大可能震荡或发散；
- 现在我们已经第一次真正让 parameter 因为训练数据而改变；
- 下一节 5.5 会把 Step 5 收尾：**到底什么叫“模型学到了东西”，以及 architecture、parameter、data、loss、optimizer 各自负责什么。**